In [0]:
%restart_python

In [0]:
%sql
  SELECT DISTINCT
    cms.Document_Id,
    cms.scheduled,
    CMS.created as created_by,
    cms.lastAuthor as last_updated_by,
    UA.event_type as Audit_event_type,
    UA.UserName as Audit_userID,
    UA.Access_TS as Audit_TS,
    UA.Events as Audit_interactions,
    CASE
      WHEN UA.event_type is null THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON trim(cms.Document_Id) = trim(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UP.BU IS NOT NULL AND lower(UP.BU) NOT in ( 'left organization', 'unidentified')
    AND UA.WEBI_Doc_ID IS NULL
;
-- select scheduled, count(distinct Document_Id) from _sqldf
-- group by scheduled
-- Total in scope: 21,986
-- Owner identified : 17,178 (13,462 active in Audit, 3,716 reports Flagged for schedules)
-- Owner unidentified: 4,808 ()

-- select count(distinct Document_Id) as cnt from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS


In [0]:
%sql

select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms

In [0]:
%sql
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.scheduled,
    CMS.created as created_by,
    cms.lastAuthor as last_updated_by,
    UA.event_type as Audit_event_type,
    UA.UserName as Audit_userID,
    UA.Access_TS as Audit_TS,
    UA.Events as Audit_interactions,
    CASE
      WHEN UA.event_type is null THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON trim(cms.Document_Id) = trim(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL or lower(UP.BU) = 'left organization' or lower(UP.BU) = 'unidentified')
    -- AND UA.WEBI_Doc_ID IS NOT NULL
), 
ad_userprofile AS(
   select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles 
   where AD_status = TRUE
),
report_stats as (
    SELECT
    ur.Document_Id,
    ur.created_by,
    ad_cb.BU                        AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU                        AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU                        AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions)      AS Audit_interactions
    FROM unowned_reports AS ur
    LEFT JOIN ad_userprofile AS ad_cb
    ON UPPER(TRIM(ur.created_by))      = UPPER(TRIM(ad_cb.user_ID))
    LEFT JOIN ad_userprofile AS ad_lu
    ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
    LEFT JOIN ad_userprofile AS ad_au
    ON UPPER(TRIM(ur.Audit_userID))    = UPPER(TRIM(ad_au.user_ID))

    GROUP BY
    ur.Document_Id, ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
)
-- ,
-- calculated_bu AS (
  SELECT
    Document_Id,
    created_by_BU,
    last_updated_by_BU,
    Audit_userID_BU,
    usage_category,
    total_Audit_interactions,
    CASE
      WHEN Audit_userID_BU IS NOT NULL THEN Audit_userID_BU
      WHEN last_updated_by_BU IS NOT NULL THEN last_updated_by_BU
      ELSE created_by_BU
    END AS Recommended_BU_owner,
    CASE
      WHEN Audit_userID_BU    IS NOT NULL THEN 'Audit interaction'
      WHEN last_updated_by_BU IS NOT NULL or created_by_BU      IS NOT NULL THEN 'Last user updated Report from BU'
      -- WHEN  THEN 'Last user updated Report from BU'
      ELSE 'Obsolete system records'
    END AS Recommend_reason
  FROM (
    SELECT
      Document_Id,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY 
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY
      Document_Id,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category
  )
  WHERE rn = 1
-- )
-- select cbu1.document_id, cbu1.recommended_bu_owner, cbu1.Recommend_reason, cms2.Document_name, cms2.Full_path, cms2.scheduled, cms2.updated as lastupdat_TS from calculated_bu AS cbu1
-- left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS cms2
-- on trim(cbu1.Document_Id) = trim(cms2.Document_Id)
-- where cbu1.recommended_bu_owner is not null


In [0]:
import matplotlib.pyplot as plt

# Materialise once from cell 2 SQL result
df_reason_pd = _sqldf.toPandas()

# Deduplicate: one row per Document_Id using explicit priority
_PRIORITY = {
    'Audit interaction': 1,
    'Last user updated Report from BU':                 2,
    'Created by user from BU':           3,
    'Obsolete system records':           4,
}
df_reason_pd['_priority'] = df_reason_pd['Recommend_reason'].map(_PRIORITY).fillna(99)
df_dedup = (
    df_reason_pd
    .sort_values('_priority')
    .drop_duplicates('Document_Id', keep='first')
    .drop(columns='_priority')
    .reset_index(drop=True)
)

# --- Aggregations ---
# Break down 'Audit interaction' by usage_category (Editor/Viewer)
def assign_pie_label(r):
    if r['Recommend_reason'] == 'Audit interaction':
        return f"Audit interaction ({r['usage_category'].replace('_events', '')})"
    else:
        return r['Recommend_reason']

df_dedup['Pie_label'] = df_dedup.apply(assign_pie_label, axis=1)

pdf = (
    df_dedup
    .groupby('Pie_label', as_index=False)['Document_Id']
    .count()
    .rename(columns={'Document_Id': 'Document_Count', 'Pie_label': 'Recommend_reason'})
)
# Keep Obsolete slices together, then Audit together, then others
_GROUP_ORDER = {
    'Obsolete system records': 1,
    'Audit interaction (Editor)': 2,
    'Audit interaction (Viewer)': 2,
    'Last user updated Report from BU': 3,
}
pdf['_grp'] = pdf['Recommend_reason'].map(_GROUP_ORDER).fillna(99)
pdf = pdf.sort_values(['_grp', 'Document_Count'], ascending=[True, False]).drop(columns='_grp').reset_index(drop=True)

pdf_owner = (
    df_dedup[df_dedup['Recommended_BU_owner'].notna()]
    .groupby('Recommended_BU_owner', as_index=False)['Document_Id']
    .count()
    .rename(columns={'Document_Id': 'Document_Count'})
    .sort_values('Document_Count', ascending=False)
)

# --- Chart 1: Pie — Document count by Recommend Reason ---
display(pdf)
fig1, ax1 = plt.subplots(figsize=(7, 7))
color_map = {
    'Obsolete system records': 'blue',
    'Last user updated Report from BU': 'orange',
    'Audit interaction (Editor)': 'green',
    'Audit interaction (Viewer)': 'lightgreen',
}
colors = [color_map.get(r, 'grey') for r in pdf['Recommend_reason']]
total = pdf['Document_Count'].sum()
def make_autopct(values):
    def autopct(pct):
        count = int(round(pct * total / 100.0))
        return f'{count:,} | {pct:.1f}%'
    return autopct

wedges, texts, autotexts = ax1.pie(
    pdf['Document_Count'],
    labels=pdf['Recommend_reason'],
    autopct=make_autopct(pdf['Document_Count']),
    startangle=140,
    colors=colors,
    pctdistance=0.75,
)
# Push 'Audit interaction (Viewer)' label outward to avoid overlap
for i, label in enumerate(pdf['Recommend_reason']):
    if label == 'Audit interaction (Viewer)':
        x, y = autotexts[i].get_position()
        autotexts[i].set_position((x * 1.2, y * 1.2))
ax1.set_title('Document Count by Recommend Reason')
plt.tight_layout()
plt.show()

# --- Chart 2: Bar — Document count by Recommended BU Owner ---
display(pdf_owner)
fig2, ax2 = plt.subplots(figsize=(10, 6))
ax2.bar(pdf_owner['Recommended_BU_owner'], pdf_owner['Document_Count'], color='steelblue')
ax2.set_xlabel('Recommended BU Owner')
ax2.set_ylabel('Document Count')
ax2.set_title('Document Count by Recommended BU Owner')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [0]:
%sql
-- Obsolete system records — no recommended owner
-- Uses same ROW_NUMBER dedup as cell 4 to match totals
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.scheduled,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.scheduled,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.scheduled, ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      scheduled,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, scheduled, created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT Document_Id, scheduled
  FROM deduped
  WHERE Audit_userID_BU IS NULL
    AND last_updated_by_BU IS NULL
    AND created_by_BU IS NULL
)
SELECT
  CASE
    WHEN oi.scheduled = TRUE AND sd.schedule_activeness = 'Active'   THEN 'Scheduled - Active'
    WHEN oi.scheduled = TRUE AND sd.schedule_activeness = 'Inactive' THEN 'Scheduled - Inactive'
    WHEN oi.scheduled = TRUE AND sd.schedule_activeness IS NULL      THEN 'No schedule details found'
    ELSE 'Not Scheduled'
  END AS schedule_category,
  COUNT(DISTINCT oi.Document_Id) AS Document_Count
FROM obsolete_records AS oi
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_active AS sd
  ON TRIM(oi.Document_Id) = TRIM(sd.document_id)
GROUP BY 1
ORDER BY Document_Count DESC

In [0]:
%sql
-- Obsolete system records: categorise by folder location
-- Uses same ROW_NUMBER dedup as cell 4 to match totals
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Full_path,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.Full_path,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.Full_path, ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      Full_path,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, Full_path, created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT Document_Id, Full_path
  FROM deduped
  WHERE Audit_userID_BU IS NULL
    AND last_updated_by_BU IS NULL
    AND created_by_BU IS NULL
)
SELECT
  CASE
    -- Public folders in any language
    WHEN LOWER(Full_path) LIKE '%public%'
      OR LOWER(Full_path) LIKE '%openbaar%'
      OR LOWER(Full_path) LIKE '%publiek%'
      OR LOWER(Full_path) LIKE '%publique%'
      OR LOWER(Full_path) LIKE '%öffentlich%'
      OR LOWER(Full_path) LIKE '%público%'
      OR LOWER(Full_path) LIKE '%pubblic%'
    THEN 'Public'
    -- Private folders (multi-language)
    WHEN LOWER(Full_path) LIKE '%my folders%'
      OR LOWER(Full_path) LIKE '%my box%'
      OR LOWER(Full_path) LIKE '%inbox%'
      OR LOWER(Full_path) LIKE '%favorites%'
      OR LOWER(Full_path) LIKE '%personal%'
      OR LOWER(Full_path) LIKE '%meine ordner%'
      OR LOWER(Full_path) LIKE '%posteingang%'
    THEN 'Private'
    ELSE 'Other'
  END AS folder_category,
  COUNT(DISTINCT Document_Id) AS Document_Count
FROM obsolete_records
GROUP BY 1
ORDER BY Document_Count DESC

In [0]:
%sql
-- Obsolete system records in Public folders: breakdown by subfolder path
-- Uses same ROW_NUMBER dedup as cell 4 to match totals
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Full_path,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.Full_path,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.Full_path, ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      Full_path,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, Full_path, created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT Document_Id, Full_path
  FROM deduped
  WHERE Audit_userID_BU IS NULL
    AND last_updated_by_BU IS NULL
    AND created_by_BU IS NULL
),
public_reports AS (
  SELECT
    Document_Id,
    Full_path
  FROM obsolete_records
  WHERE (LOWER(Full_path) LIKE '%public%'
    OR LOWER(Full_path) LIKE '%openbaar%'
    OR LOWER(Full_path) LIKE '%publiek%'
    OR LOWER(Full_path) LIKE '%publique%'
    OR LOWER(Full_path) LIKE '%öffentlich%'
    OR LOWER(Full_path) LIKE '%público%'
    OR LOWER(Full_path) LIKE '%pubblic%')
    AND NOT (
      LOWER(Full_path) LIKE '%meine ordner%'
      OR LOWER(Full_path) LIKE '%posteingang%'
    )
)
SELECT
  -- Show first 3 folder levels for grouping
  CONCAT_WS('/',
    SPLIT(Full_path, '/')[0],
    SPLIT(Full_path, '/')[1],
    get(SPLIT(Full_path, '/'), 2)
  ) AS folder_path,
  COUNT(DISTINCT Document_Id) AS Document_Count
FROM public_reports
GROUP BY 1
ORDER BY Document_Count DESC

In [0]:
%sql
-- Obsolete system records: analyse the 'created' field BU values
-- Uses same ROW_NUMBER dedup as cell 4 to match totals
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Full_path,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.Full_path,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.Full_path, ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      Full_path,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, Full_path, created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT DISTINCT d.Document_Id, d.Full_path, rs.created_by
  FROM deduped d
  JOIN report_stats rs ON d.Document_Id = rs.Document_Id
  WHERE d.Audit_userID_BU IS NULL
    AND d.last_updated_by_BU IS NULL
    AND d.created_by_BU IS NULL
)
SELECT
  CASE
    WHEN LOWER(TRIM(oi.created_by)) = 'administrator' THEN 'SAP BO Admin'
    WHEN UP.BU IS NULL or LOWER(UP.BU) in('left organization', 'unidentified') THEN 'No profile found'
    WHEN oi.created_by IS NULL THEN 'Blank field in CMS'
    ELSE UP.BU
  END AS creator_Group,
  COUNT(DISTINCT oi.Document_Id) AS Document_Count
FROM obsolete_records oi
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(oi.created_by)) = UPPER(TRIM(UP.UserName))
GROUP BY creator_Group
ORDER BY Document_Count DESC

In [0]:
%sql
-- List Document CUID, Name, and Full Path for obsolete reports in Public folders
-- Uses same ROW_NUMBER dedup as cell 4 to match totals
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Document_CUID,
    cms.Document_name,
    cms.Full_path,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.Document_CUID,
    ur.Document_name,
    ur.Full_path,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.Document_CUID, ur.Document_name, ur.Full_path,
    ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      Document_CUID,
      Document_name,
      Full_path,
      created_by AS user_ID,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, Document_CUID, Document_name, Full_path,
      created_by,
      created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT Document_Id, Document_CUID, Document_name, Full_path, user_ID
  FROM deduped
  WHERE Audit_userID_BU IS NULL
    AND last_updated_by_BU IS NULL
    AND created_by_BU IS NULL
)
SELECT DISTINCT
  Document_CUID,
  Document_name,
  Full_path,
  user_ID
FROM obsolete_records
WHERE (LOWER(Full_path) LIKE '%public%'
  OR LOWER(Full_path) LIKE '%openbaar%'
  OR LOWER(Full_path) LIKE '%publiek%'
  OR LOWER(Full_path) LIKE '%publique%'
  OR LOWER(Full_path) LIKE '%öffentlich%'
  OR LOWER(Full_path) LIKE '%público%'
  OR LOWER(Full_path) LIKE '%pubblic%')
  AND NOT (
    LOWER(Full_path) LIKE '%meine ordner%'
    OR LOWER(Full_path) LIKE '%posteingang%'
  )
ORDER BY Full_path, Document_name

In [0]:
%sql
-- Detail of obsolete reports where CMS 'created' field is blank
-- Uses same ROW_NUMBER dedup as cell 4
WITH unowned_reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    cms.Document_CUID,
    cms.Document_name,
    cms.Full_path,
    CMS.created AS created_by,
    cms.lastAuthor AS last_updated_by,
    UA.UserName AS Audit_userID,
    UA.Events AS Audit_interactions,
    CASE
      WHEN UA.event_type IS NULL THEN 'Audit_Inactive'
      WHEN UA.event_type IN ('Edit','Deliver','Modify','Save','Delete','Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON TRIM(cms.Document_Id) = TRIM(UA.WEBI_Doc_ID)
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND (UP.BU IS NULL OR LOWER(UP.BU) IN ('left organization','unidentified'))
),
ad_userprofile AS (
  SELECT * FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
  WHERE AD_status = TRUE
),
report_stats AS (
  SELECT
    ur.Document_Id,
    ur.Document_CUID,
    ur.Document_name,
    ur.Full_path,
    ur.created_by,
    ad_cb.BU AS created_by_BU,
    ur.last_updated_by,
    ad_lu.BU AS last_updated_by_BU,
    ur.Audit_userID,
    ad_au.BU AS Audit_userID_BU,
    ur.usage_category,
    SUM(ur.Audit_interactions) AS Audit_interactions
  FROM unowned_reports AS ur
  LEFT JOIN ad_userprofile AS ad_cb ON UPPER(TRIM(ur.created_by)) = UPPER(TRIM(ad_cb.user_ID))
  LEFT JOIN ad_userprofile AS ad_lu ON UPPER(TRIM(ur.last_updated_by)) = UPPER(TRIM(ad_lu.user_ID))
  LEFT JOIN ad_userprofile AS ad_au ON UPPER(TRIM(ur.Audit_userID)) = UPPER(TRIM(ad_au.user_ID))
  GROUP BY
    ur.Document_Id, ur.Document_CUID, ur.Document_name, ur.Full_path,
    ur.created_by, ad_cb.BU,
    ur.last_updated_by, ad_lu.BU,
    ur.Audit_userID, ad_au.BU,
    ur.usage_category
),
deduped AS (
  SELECT *
  FROM (
    SELECT
      Document_Id,
      Document_CUID,
      Document_name,
      Full_path,
      created_by,
      created_by_BU,
      last_updated_by_BU,
      Audit_userID_BU,
      usage_category,
      SUM(Audit_interactions) AS total_Audit_interactions,
      ROW_NUMBER() OVER (
        PARTITION BY Document_Id
        ORDER BY
          CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 2 END,
          SUM(Audit_interactions) DESC,
          CASE WHEN Audit_userID_BU IS NOT NULL THEN 1 ELSE 2 END
      ) AS rn
    FROM report_stats
    GROUP BY Document_Id, Document_CUID, Document_name, Full_path,
      created_by, created_by_BU, last_updated_by_BU, Audit_userID_BU, usage_category
  )
  WHERE rn = 1
),
obsolete_records AS (
  SELECT Document_Id, Document_CUID, Document_name, Full_path, created_by
  FROM deduped
  WHERE Audit_userID_BU IS NULL
    AND last_updated_by_BU IS NULL
    AND created_by_BU IS NULL
)
SELECT DISTINCT
  o.Document_CUID,
  o.Document_name,
  o.Full_path,
  o.created_by
FROM obsolete_records o
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(o.created_by)) = UPPER(TRIM(UP.UserName))
WHERE LOWER(TRIM(o.created_by)) != 'administrator'
  AND UP.BU IS NULL
ORDER BY o.Full_path, o.Document_name